`requirements/runtime.txt`

In [3]:
import jsonpickle, os, csv
from datetime import datetime, UTC, timezone
from dotenv import load_dotenv

from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime import QiskitRuntimeService

In [12]:
load_dotenv()

service = QiskitRuntimeService.save_account(
    token=os.getenv("QISKIT_IBM_TOKEN"), 
    instance=os.getenv("QISKIT_IBM_INSTANCE"),
    overwrite=True
)

In [ ]:
backends = ["ibm_fez", "ibm_marrakesh", "ibm_torino"]
service = QiskitRuntimeService()

for backend_name in backends:
    backend = service.backend(backend_name)
    noise_model = NoiseModel.from_backend(backend)

    payload = {
        "noise_model": noise_model.to_dict(),
        "coupling_map": backend.configuration().coupling_map,
        
    }

    ts = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")    
    backend_dir = os.path.join("calibrations", backend_name)
    os.makedirs(backend_dir, exist_ok=True)

    path = os.path.join(backend_dir, f"{ts}.json")
    with open(path, "w") as f:
        f.write(jsonpickle.encode(payload, indent=2))


In [6]:
service = QiskitRuntimeService()
backend = service.backend("ibm_marrakesh", use_fractional_gates=True)

dt = datetime(2026, 1, 29, 8, 16, 31, tzinfo=timezone.utc)
props = backend.properties(datetime=dt)
target = backend.target
coupling_map = backend.configuration().coupling_map or []

out_dir = "calibrations"
os.makedirs(out_dir, exist_ok=True)

out_path_1q = os.path.join(out_dir, "errors.csv")
out_path_2q = os.path.join(out_dir, "errors_2_qubits.csv")


def qprop(q, name):
    d = props.qubit_property(q)
    return d.get(name, (None,))[0]


def instr_error(name, qargs):
    try:
        inst = target[name][qargs]
        return None if inst is None else inst.error
    except Exception:
        return None


def instr_duration_ns(name, qargs):
    try:
        inst = target[name][qargs]
        if inst is None or inst.duration is None:
            return None
        return inst.duration * 1e9
    except Exception:
        return None


# ---------- 1Q / qubit-level file ----------
fieldnames_1q = [
    "Qubit",
    "T1 (us)",
    "T2 (us)",
    "Readout assignment error",
    "Prob meas0 prep1",
    "Prob meas1 prep0",
    "Readout length (ns)",
    "ID error",
    "Single-qubit gate length (ns)",
    "RX error",
    "Z-axis rotation (rz) error",
    "√x (sx) error",
    "Pauli-X error",
    "MEASURE error",
    "Operational",
]

with open(out_path_1q, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames_1q)
    writer.writeheader()

    for q in range(backend.num_qubits):
        t1 = qprop(q, "T1")
        t2 = qprop(q, "T2")
        readout_len = qprop(q, "readout_length")

        row = {
            "Qubit": q,
            "T1 (us)": None if t1 is None else t1 * 1e6,
            "T2 (us)": None if t2 is None else t2 * 1e6,
            "Readout assignment error": qprop(q, "readout_error"),
            "Prob meas0 prep1": qprop(q, "prob_meas0_prep1"),
            "Prob meas1 prep0": qprop(q, "prob_meas1_prep0"),
            "Readout length (ns)": None if readout_len is None else readout_len * 1e9,
            "ID error": instr_error("id", (q,)),
            "Single-qubit gate length (ns)": instr_duration_ns("id", (q,)),
            "RX error": instr_error("rx", (q,)),
            "Z-axis rotation (rz) error": instr_error("rz", (q,)),
            "√x (sx) error": instr_error("sx", (q,)),
            "Pauli-X error": instr_error("x", (q,)),
            "MEASURE error": instr_error("measure", (q,)),
            "Operational": "Yes",
        }

        writer.writerow(row)


# ---------- 2Q / edge-level file ----------
fieldnames_2q = [
    "Qubit 1",
    "Qubit 2",
    "CZ error",
    "Gate length (ns)",
    "RZZ error",
    "Operational",
]

with open(out_path_2q, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames_2q)
    writer.writeheader()

    for q1, q2 in coupling_map:
        row = {
            "Qubit 1": q1,
            "Qubit 2": q2,
            "CZ error": instr_error("cz", (q1, q2)),
            "Gate length (ns)": instr_duration_ns("cz", (q1, q2)),
            "RZZ error": instr_error("rzz", (q1, q2)),
            "Operational": "Yes",
        }
        writer.writerow(row)

print(f"Saved {out_path_1q}")
print(f"Saved {out_path_2q}")

Saved calibrations/errors.csv
Saved calibrations/errors_2_qubits.csv
